<a href="https://colab.research.google.com/github/NSmonish/Edge-Aware_LoRA-MoE_AI4Dev26/blob/main/EdgeAwareLoRA_MoE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Edge-Aware LoRA-MoE for mnli roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import random
from collections import defaultdict
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()

        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)

        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()

        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))

        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)

        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)

        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros(
            (batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device
        )

        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))

        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling

        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(
                    selected_experts.reshape(-1), minlength=self.num_experts
                ).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None

        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    """ Custom Trainer to incorporate the load balancing loss. """
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Pass kwargs to the parent class to handle new arguments
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if isinstance(m, EdgeAwareLoRALinear):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    """ Replaces target linear layers in RoBERTa with our custom MoE LoRA layer. """
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model


def compute_metrics(eval_pred):
    """ Calculates evaluation metrics (accuracy, F1, etc.). """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
print("Loading and preparing dataset...")
dataset = load_dataset("glue", "mnli")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(example):
    return tokenizer(example["premise"], example["hypothesis"], truncation=True, padding="max_length", max_length=128)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"]
eval_dataset = dataset["validation_matched"]
print("Dataset preparation finished.")

In [ ]:
seeds = [42, 123, 7, 99, 101]
all_results = []

for seed in seeds:
    print(f"\n{'='*30}\n  STARTING RUN FOR SEED: {seed}\n{'='*30}\n")

    set_seed(seed)
    out_dir = f"./ourlora_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    # --- Model Initialization (Fresh model for each seed) ---
    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=3)
    model.roberta = patch_roberta_with_edge_aware_lora(
        model.roberta, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32
    )

    # Freeze all parameters except the new LoRA modules
    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Seed {seed} | Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    # --- Training Arguments ---
    training_args = TrainingArguments(
        output_dir=f"./edge-aware-lora-roberta-mnli-seed-{seed}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=3e-4,
        per_device_train_batch_size=128,
        per_device_eval_batch_size=128,
        num_train_epochs=10,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs/seed-{seed}",
        logging_steps=500,
        report_to="none",
        fp16=True,
        seed=seed,
    )

    # --- Trainer Initialization and Training ---
    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=5e-3,
    )

    print(f"Starting training for seed {seed}...")
    trainer.train()
    print(f"Training finished for seed {seed}.")

    # --- Post-Training Analysis & Saving ---
    print(f"\n--- Analysis and Saving for SEED: {seed} ---")
    model.save_pretrained(out_dir)

    # Generate and save learning curve plots
    log_history_df = pd.DataFrame(trainer.state.log_history)
    train_logs = log_history_df.dropna(subset=['loss'])
    eval_logs = log_history_df.dropna(subset=['eval_loss'])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f"Training Metrics for Seed {seed}", fontsize=16)

    ax1.plot(train_logs['step'], train_logs['loss'], label='Training Loss')
    ax1.plot(eval_logs['step'], eval_logs['eval_loss'], label='Validation Loss')
    ax1.set_title('Training & Validation Loss')
    ax1.set_xlabel('Steps'); ax1.set_ylabel('Loss')
    ax1.legend(); ax1.grid(True)

    ax2.plot(eval_logs['step'], eval_logs['eval_accuracy'], label='Validation Accuracy', color='green')
    ax2.set_title('Validation Accuracy')
    ax2.set_xlabel('Steps'); ax2.set_ylabel('Accuracy')
    ax2.legend(); ax2.grid(True)

    plt.tight_layout()
    plot_path = f"{out_dir}/learning_curve.png"
    plt.savefig(plot_path)
    plt.close()
    print(f"Learning curve plot saved to: {plot_path}")

    # Aggregate and save metrics to CSV
    # ... (code for creating combined_df is here) ...
    csv_path = f"{out_dir}/epoch_metrics.csv"
    # combined_df.to_csv(csv_path, index=False, float_format="%.4f")
    # For simplicity, we just save the full log history.
    log_history_df.to_csv(csv_path, index=False, float_format="%.4f")


    print(f"Results for seed {seed} saved to {out_dir}")
    print(f"\n{'='*30}\n  FINISHED RUN FOR SEED: {seed}\n{'='*30}\n")

    # Store the final eval accuracy
    final_eval_metrics = {k: v for k, v in trainer.state.log_history[-1].items() if k.startswith('eval')}
    final_eval_metrics['seed'] = seed
    all_results.append(final_eval_metrics)

In [ ]:
print("\n\n--- ALL SEED RUNS COMPLETED ---")
results_df = pd.DataFrame(all_results)
print("Summary of final evaluation accuracy across all seeds:")
print(results_df[['seed', 'eval_accuracy', 'eval_loss']])

print("\nAverage validation accuracy: {:.4f}".format(results_df['eval_accuracy'].mean()))
print("Standard deviation of validation accuracy: {:.4f}".format(results_df['eval_accuracy'].std()))
print("Peak validation accuracy:    {:.4f}".format(results_df['eval_accuracy'].max()))
print("\nExperiment finished successfully! 🚀")

# Edge-Aware LoRA-MoE for sst-2 roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import random
from collections import defaultdict
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()

        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None # Initialize lb_loss attribute

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)

        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()

        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))

        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)

        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)

        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros(
            (batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device
        )

        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))

        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling

        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(
                    selected_experts.reshape(-1), minlength=self.num_experts
                ).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None

        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    """ Custom Trainer to incorporate the load balancing loss. """
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    """ Replaces target linear layers in RoBERTa with our custom MoE LoRA layer. """
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

def compute_metrics(eval_pred):
    """ Calculates evaluation metrics (accuracy, F1, etc.). """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
print("Loading and preparing SST-2 dataset...")
dataset = load_dataset("glue", "sst2")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(example):
    return tokenizer(example["sentence"], truncation=True, padding="max_length", max_length=128)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
print("Dataset preparation finished.")

In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")

    set_seed(seed)
    # --- CHANGE 2: Simplified output directory ---
    out_dir = f"./ourlora_sst2_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    model.roberta = patch_roberta_with_edge_aware_lora(
        model.roberta, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32
    )

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    training_args = TrainingArguments(
        output_dir=f"./run_sst2_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=3e-4,
        per_device_train_batch_size=128,
        per_device_eval_batch_size=128,
        num_train_epochs=10,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs_sst2/seed{seed}",
        logging_steps=50,
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")

    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    log_history_df = pd.DataFrame(trainer.state.log_history)
    log_history_df.to_csv(f"{out_dir}/log_history.csv", index=False)

    eval_logs = log_history_df.dropna(subset=['eval_loss'])

    if not eval_logs.empty:
        final_eval_metrics = eval_logs.iloc[-1].to_dict()
        final_eval_metrics['seed'] = seed
        all_results.append(final_eval_metrics)
    else:
        print("No evaluation logs found for this run.")

    print(f"Results for this run saved to {out_dir}")

In [ ]:
print("\n\n--- ALL 5 SEED RUNS COMPLETED ---")
if all_results:
    results_df = pd.DataFrame(all_results)
    print("--- Final Evaluation Results per Seed ---")
    # Display accuracy with more precision
    print(results_df[['seed', 'eval_accuracy', 'eval_loss']].to_string(float_format="%.4f"))

    # --- CHANGE 3: Calculate and print average, best, and std dev ---
    avg_accuracy = results_df['eval_accuracy'].mean()
    best_accuracy = results_df['eval_accuracy'].max()
    std_dev_accuracy = results_df['eval_accuracy'].std()

    print("\n--- Summary Across 5 Seeds ---")
    print(f"📊 Average Accuracy: {avg_accuracy:.4f}")
    print(f"🏆 Best Accuracy:    {best_accuracy:.4f}")
    print(f"📈 Std Deviation:    {std_dev_accuracy:.4f}")
else:
    print("No results to summarize.")

print("\nExperiment finished successfully! 🚀")

# Edge-Aware LoRA-MoE for mrpc roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()

        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None # Initialize lb_loss attribute

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)

        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()

        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))

        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)

        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)

        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros(
            (batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device
        )

        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))

        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling

        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(
                    selected_experts.reshape(-1), minlength=self.num_experts
                ).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None

        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    """ Custom Trainer to incorporate the load balancing loss. """
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    """ Replaces target linear layers in RoBERTa with our custom MoE LoRA layer. """
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

def compute_metrics(eval_pred):
    """ Calculates evaluation metrics (accuracy, F1, etc.). """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
print("Loading and preparing MRPC dataset...")
dataset = load_dataset("glue", "mrpc")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True, padding="max_length", max_length=128)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"]
eval_dataset = dataset["test"]
print("Dataset preparation finished.")

In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")

    set_seed(seed)
    out_dir = f"./ourlora_mrpc_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    model.roberta = patch_roberta_with_edge_aware_lora(
        model.roberta, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32
    )

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    training_args = TrainingArguments(
        output_dir=f"./run_mrpc_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="no",
        learning_rate=3e-4,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=20,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs_mrpc/seed{seed}",
        logging_steps=20,
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")

    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    log_history_df = pd.DataFrame(trainer.state.log_history)
    eval_logs = log_history_df.dropna(subset=['eval_loss'])

    if not eval_logs.empty:
        final_eval_metrics = eval_logs.iloc[-1].to_dict()
        final_eval_metrics['seed'] = seed
        all_results.append(final_eval_metrics)
    else:
        print("No evaluation logs found for this run.")

    print(f"Results for this run saved to {out_dir}")

In [ ]:
if all_results:
    results_df = pd.DataFrame(all_results)
    print("--- Final Evaluation Results per Seed ---")
    print(results_df[['seed', 'eval_accuracy', 'eval_loss']].to_string(float_format="%.4f"))

    # --- CHANGE 3: Calculate and print average, best, and std dev ---
    avg_accuracy = results_df['eval_accuracy'].mean()
    best_accuracy = results_df['eval_accuracy'].max()
    std_dev_accuracy = results_df['eval_accuracy'].std()

    print("\n--- Summary Across 5 Seeds on MRPC ---")
    print(f"📊 Average Accuracy: {avg_accuracy:.4f}")
    print(f"🏆 Best Accuracy:    {best_accuracy:.4f}")
    print(f"📈 Std Deviation:    {std_dev_accuracy:.4f}")
else:
    print("No results to summarize.")

print("\nExperiment finished successfully! 🚀")

# Edge-Aware LoRA-MoE for CoLA roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, matthews_corrcoef
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()

        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None # Initialize lb_loss attribute

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)

        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()

        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))

        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)

        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)

        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros(
            (batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device
        )

        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))

        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling

        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(
                    selected_experts.reshape(-1), minlength=self.num_experts
                ).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None

        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    """ Custom Trainer to incorporate the load balancing loss. """
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    """ Replaces target linear layers in RoBERTa with our custom MoE LoRA layer. """
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

# --- Add Matthews Correlation Coefficient (MCC) for CoLA ---
def compute_metrics(eval_pred):
    """ Calculates evaluation metrics, including MCC for CoLA. """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    mcc = matthews_corrcoef(labels, preds)
    return {"accuracy": acc, "mcc": mcc}

In [ ]:
print("Loading and preparing CoLA dataset...")
dataset = load_dataset("glue", "cola")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(example):
    return tokenizer(example["sentence"], truncation=True, padding="max_length", max_length=128)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# CoLA has official train/validation splits
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
print("Dataset preparation finished.")

In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")

    set_seed(seed)
    # --- CHANGE 4: Update output directories for CoLA ---
    out_dir = f"./ourlora_cola_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    # num_labels=2 is correct for CoLA (acceptable or not)
    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    model.roberta = patch_roberta_with_edge_aware_lora(
        model.roberta, num_experts=4, top_k=2, shared_rank=4, lora_alpha=4
    )

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    training_args = TrainingArguments(
        output_dir=f"./run_cola_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="mcc",
        learning_rate=2e-4,
        per_device_train_batch_size=64,
        per_device_eval_batch_size=64,
        num_train_epochs=20,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs_cola/seed{seed}",
        logging_steps=50,
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")

    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    # Evaluate the best model on the validation set
    final_metrics = trainer.evaluate()
    final_metrics['seed'] = seed
    all_results.append(final_metrics)

In [ ]:
if all_results:
    results_df = pd.DataFrame(all_results)
    print("--- Final Evaluation Results per Seed ---")
    # --- CHANGE 6: Display eval_mcc in the summary table ---
    print(results_df[['seed', 'eval_accuracy', 'eval_mcc', 'eval_loss']].to_string(float_format="%.4f"))

    avg_mcc = results_df['eval_mcc'].mean()
    best_mcc = results_df['eval_mcc'].max()
    std_dev_mcc = results_df['eval_mcc'].std()

    avg_accuracy = results_df['eval_accuracy'].mean()
    best_accuracy = results_df['eval_accuracy'].max()

    print("\n--- Summary Across 5 Seeds on CoLA ---")
    print(f"🎯 Average MCC: {avg_mcc:.4f}")
    print(f"🏆 Best MCC:    {best_mcc:.4f}")
    print(f"📈 Std Dev MCC: {std_dev_mcc:.4f}")
    print("---")
    print(f"📊 Average Accuracy: {avg_accuracy:.4f}")
    print(f"✨ Best Accuracy:    {best_accuracy:.4f}")
else:
    print("No results to summarize.")

print("\nExperiment finished successfully! 🚀")

# Edge-Aware LoRA-MoE for STS-B roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from scipy.stats import pearsonr, spearmanr
import os

if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)
        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()
        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))
        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)
        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)
        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros((batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device)
        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))
        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling
        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(selected_experts.reshape(-1), minlength=self.num_experts).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None
        return original_output + final_lora_output


In [ ]:
class LBTrainer(Trainer):
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.squeeze(logits) # Squeeze predictions for regression
    pearson_corr, _ = pearsonr(preds, labels)
    spearman_corr, _ = spearmanr(preds, labels)
    return {
        "pearson": pearson_corr,
        "spearmanr": spearman_corr,
    }

In [ ]:
print("Loading and preparing STS-B dataset...")
dataset = load_dataset("glue", "stsb")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True, padding="max_length", max_length=512)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
print("Dataset preparation finished.")

In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")
    set_seed(seed)
    out_dir = f"./ourlora_stsb_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=1)
    model.roberta = patch_roberta_with_edge_aware_lora(model.roberta, num_experts=4, top_k=2, shared_rank=4, lora_alpha=4)

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    training_args = TrainingArguments(
        output_dir=f"./run_stsb_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="pearson",
        learning_rate=2e-4,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        num_train_epochs=30,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs_stsb/seed{seed}",
        logging_steps=50,
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")
    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    final_metrics = trainer.evaluate()
    final_metrics['seed'] = seed
    all_results.append(final_metrics)


In [ ]:
if all_results:
    results_df = pd.DataFrame(all_results)
    print("--- Final Evaluation Results per Seed ---")
    # --- CHANGE 7: Display regression metrics in the summary table ---
    print(results_df[['seed', 'eval_pearson', 'eval_spearmanr', 'eval_loss']].to_string(float_format="%.4f"))

    avg_pearson = results_df['eval_pearson'].mean()
    best_pearson = results_df['eval_pearson'].max()
    std_dev_pearson = results_df['eval_pearson'].std()

    print("\n--- Summary Across 5 Seeds on STS-B ---")
    print(f"🎯 Average Pearson: {avg_pearson:.4f}")
    print(f"🏆 Best Pearson:    {best_pearson:.4f}")
    print(f"📈 Std Dev Pearson: {std_dev_pearson:.4f}")
else:
    print("No results to summarize.")

print("\nExperiment finished successfully! 🚀")

# Edge-Aware LoRA-MoE for RTE roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)
        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()
        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))
        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)
        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)
        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros((batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device)
        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))
        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling
        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(selected_experts.reshape(-1), minlength=self.num_experts).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None
        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

# --- CHANGE 1: Update metrics function for classification (Accuracy) ---
def compute_metrics(eval_pred):
    """ Calculates evaluation metrics for classification. """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

In [ ]:
print("Loading and preparing RTE dataset...")
dataset = load_dataset("glue", "rte")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

# RTE uses two sentences
def tokenize(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True, padding="max_length", max_length=512) # Increased max_length for RTE

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
print("Dataset preparation finished.")

In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")
    set_seed(seed)
    # --- CHANGE 3: Update output directories for RTE ---
    out_dir = f"./ourlora_rte_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    # --- CHANGE 4: Set num_labels=2 for binary classification ---
    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    model.roberta = patch_roberta_with_edge_aware_lora(model.roberta, num_experts=4, top_k=2, shared_rank=4, lora_alpha=4)

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    # --- CHANGE 5: Adjust training arguments for a very small dataset ---
    training_args = TrainingArguments(
        output_dir=f"./run_rte_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        learning_rate=4e-4, # Lower learning rate for stability
        per_device_train_batch_size=32, # Small batch size
        per_device_eval_batch_size=64,
        num_train_epochs=20, # Train for more epochs, but load the best model
        weight_decay=0.01,
        warmup_ratio=0.1, # Larger warmup for stability
        lr_scheduler_type="linear",
        logging_dir=f"./logs_rte/seed{seed}",
        logging_steps=20,
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")
    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    final_metrics = trainer.evaluate()
    final_metrics['seed'] = seed
    all_results.append(final_metrics)

In [ ]:
if all_results:
    results_df = pd.DataFrame(all_results)
    print("--- Final Evaluation Results per Seed ---")
    # --- CHANGE 6: Display accuracy in the summary table ---
    print(results_df[['seed', 'eval_accuracy', 'eval_loss']].to_string(float_format="%.4f"))

    avg_accuracy = results_df['eval_accuracy'].mean()
    best_accuracy = results_df['eval_accuracy'].max()
    std_dev_accuracy = results_df['eval_accuracy'].std()

    print("\n--- Summary Across 5 Seeds on RTE ---")
    print(f"📊 Average Accuracy: {avg_accuracy:.4f}")
    print(f"🏆 Best Accuracy:    {best_accuracy:.4f}")
    print(f"📈 Std Deviation:    {std_dev_accuracy:.4f}")
else:
    print("No results to summarize.")

print("\nExperiment finished successfully! 🚀")

# Edge-Aware LoRA-MoE for QNLI roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)
        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()
        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))
        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)
        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)
        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros((batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device)
        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))
        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling
        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(selected_experts.reshape(-1), minlength=self.num_experts).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None
        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss

In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

def compute_metrics(eval_pred):
    """ Calculates evaluation metrics for classification. """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

In [ ]:
print("Loading and preparing QNLI dataset...")
dataset = load_dataset("glue", "qnli")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

def tokenize(example):
    return tokenizer(example["question"], example["sentence"], truncation=True, padding="max_length", max_length=512)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
print("Dataset preparation finished.")

In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")
    set_seed(seed)
    # --- CHANGE 3: Update output directories for QNLI ---
    out_dir = f"./ourlora_qnli_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    # num_labels=2 is correct for QNLI (entailment or not)
    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    model.roberta = patch_roberta_with_edge_aware_lora(model.roberta, num_experts=4, top_k=2, shared_rank=8, lora_alpha=16)

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    # --- CHANGE 4: Adjust training arguments for a larger dataset ---
    training_args = TrainingArguments(
        output_dir=f"./run_qnli_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        learning_rate=2e-4,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        num_train_epochs=10,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs_qnli/seed{seed}",
        logging_steps=200, # Log less frequently
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")
    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    final_metrics = trainer.evaluate()
    final_metrics['seed'] = seed
    all_results.append(final_metrics)

In [ ]:
import pandas as pd
import numpy as np

# Data extracted from the training logs
per_seed_data = [
    {'seed': 42, 'eval_accuracy': 0.941973, 'eval_loss': 0.183380},
    {'seed': 123, 'eval_accuracy': 0.944719, 'eval_loss': 0.178157},
    {'seed': 7, 'eval_accuracy': 0.940509, 'eval_loss': 0.179682},
    {'seed': 99, 'eval_accuracy': 0.945085, 'eval_loss': 0.174483},
    {'seed': 101, 'eval_accuracy': 0.946183, 'eval_loss': 0.177348}
]

# Create a DataFrame for easy viewing
results_df = pd.DataFrame(per_seed_data)

# Calculate summary statistics from the final epoch results
avg_accuracy = results_df['eval_accuracy'].mean()
best_final_epoch_accuracy = results_df['eval_accuracy'].max()
std_dev_accuracy = results_df['eval_accuracy'].std()

# This was the best accuracy found across any epoch of any seed
global_best_accuracy = 0.947282

# --- Print the Final Summary ---
print("--- Final Evaluation Results per Seed (from last epoch) ---")
print(results_df.to_string(index=False, float_format="%.4f"))

print("\n--- Summary Across 5 Seeds (based on final epoch) ---")
print(f"📊 Average Accuracy: {avg_accuracy:.4f}")
print(f"🏆 Best Final-Epoch Accuracy: {best_final_epoch_accuracy:.4f}")
print(f"🏆 Best Epoch Accuracy: {global_best_accuracy:.4f}")
print(f"📈 Std Deviation: {std_dev_accuracy:.4f}")
print(f"👑 Absolute Best Accuracy (any seed, any epoch): {global_best_accuracy:.4f}")

# Edge-Aware LoRA-MoE for QQP roberta-large

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset
from sklearn.metrics import accuracy_score
import os

# Enable memory-efficient Scaled Dot-Product Attention (SDPA) if using PyTorch 2.0+
if torch.cuda.is_available():
    try:
        import packaging.version as V
        if V.Version(torch.__version__) >= V.Version("2.0"):
            torch.backends.cuda.enable_mem_efficient_sdp(True)
            print("Enabled Scaled Dot-Product Attention (SDPA).")
    except Exception:
        pass

In [ ]:
class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    It replaces a standard linear layer, using a router to select expert pathways.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)
        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()
        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))
        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)
        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)
        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros((batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device)
        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0:
                continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))
        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling
        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(selected_experts.reshape(-1), minlength=self.num_experts).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None
        return original_output + final_lora_output

In [ ]:
class LBTrainer(Trainer):
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss


In [ ]:
def patch_roberta_with_edge_aware_lora(model, **kwargs):
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs)
    return model

# --- CHANGE 1: Update metrics to include F1 score for QQP ---
def compute_metrics(eval_pred):
    """ Calculates evaluation metrics for classification, including F1 score. """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [ ]:
# --- CHANGE 2: Load the QQP dataset ---
print("Loading and preparing QQP dataset...")
dataset = load_dataset("glue", "qqp")
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

# --- CHANGE 3: Update tokenize function for question1/question2 pair ---
def tokenize(example):
    return tokenizer(example["question1"], example["question2"], truncation=True, padding="max_length", max_length=512)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]
print("Dataset preparation finished.")

In [ ]:
seeds = [42, 123, 7, 99, 101]
lb_coef = 1e-2
all_results = []

In [ ]:
for seed in seeds:
    print(f"\n{'='*40}\n  STARTING RUN: SEED={seed}, LB_COEF={lb_coef}\n{'='*40}\n")
    set_seed(seed)
    # --- CHANGE 4: Update output directories for QQP ---
    out_dir = f"./ourlora_qqp_results/seed{seed}"
    os.makedirs(out_dir, exist_ok=True)

    # num_labels=2 is correct for QQP (duplicate or not)
    model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)
    model.roberta = patch_roberta_with_edge_aware_lora(model.roberta, num_experts=4, top_k=2, shared_rank=4, lora_alpha=4)

    trainable_modules = ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]
    for name, param in model.named_parameters():
        if not any(trainable_module in name for trainable_module in trainable_modules):
            param.requires_grad = False
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Trainable Params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")

    # --- CHANGE 5: Adjust training arguments for a very large dataset ---
    training_args = TrainingArguments(
        output_dir=f"./run_qqp_results/seed{seed}",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1", # F1 score is a key metric for QQP
        learning_rate=3e-4,
        per_device_train_batch_size=128,
        per_device_eval_batch_size=128,
        num_train_epochs=20,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_dir=f"./logs_qqp/seed{seed}",
        logging_steps=500, # Log much less frequently for a huge dataset
        report_to="none",
        fp16=True,
        seed=seed,
    )

    trainer = LBTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        compute_metrics=compute_metrics,
        lb_coef=lb_coef,
    )

    print(f"Starting training...")
    trainer.train()
    print(f"Training finished.")
    print(f"\n--- Analysis and Saving for SEED={seed} ---")

    final_metrics = trainer.evaluate()
    final_metrics['seed'] = seed
    all_results.append(final_metrics)

In [ ]:
if all_results:
    results_df = pd.DataFrame(all_results)
    print("--- Final Evaluation Results per Seed ---")
    # --- CHANGE 6: Display F1 score in the summary table ---
    print(results_df[['seed', 'eval_accuracy', 'eval_f1', 'eval_loss']].to_string(float_format="%.4f"))

    avg_f1 = results_df['eval_f1'].mean()
    best_f1 = results_df['eval_f1'].max()
    std_dev_f1 = results_df['eval_f1'].std()

    avg_accuracy = results_df['eval_accuracy'].mean()
    best_accuracy = results_df['eval_accuracy'].max()

    print("\n--- Summary Across 5 Seeds on QQP ---")
    print(f"🎯 Average F1 Score: {avg_f1:.4f}")
    print(f"🏆 Best F1 Score:    {best_f1:.4f}")
    print(f"📈 Std Dev F1 Score: {std_dev_f1:.4f}")
    print("---")
    print(f"📊 Average Accuracy: {avg_accuracy:.4f}")
    print(f"✨ Best Accuracy:    {best_accuracy:.4f}")
else:
    print("No results to summarize.")

print("\nExperiment finished successfully! 🚀")

# Application of Edge Aware LoRA - MoE


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score


class EdgeAwareLoRALinear(nn.Module):
    """
    Custom PyTorch module implementing a Mixture of Experts (MoE) LoRA.
    """
    def __init__(self, original_layer, num_experts=4, top_k=2, shared_rank=4, expert_hidden_dim=32, lora_alpha=4, dropout=0.1):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.shared_rank = shared_rank
        self.expert_hidden_dim = expert_hidden_dim
        self.lora_alpha = lora_alpha
        self.scaling = self.lora_alpha / self.shared_rank
        self.original_layer = original_layer
        self._lb_loss = None
        in_features = original_layer.in_features
        out_features = original_layer.out_features
        self.router = nn.Linear(in_features, self.num_experts, bias=False)
        self.lora_A_shared = nn.Linear(in_features, self.shared_rank, bias=False)
        self.lora_B_experts = nn.ModuleDict()
        self.lora_A_experts = nn.ModuleDict()
        for i in range(self.num_experts):
            expert_name = str(i)
            self.lora_B_experts[expert_name] = nn.Linear(self.shared_rank, self.expert_hidden_dim, bias=False)
            self.lora_A_experts[expert_name] = nn.Linear(self.expert_hidden_dim, self.shared_rank, bias=False)
            nn.init.kaiming_uniform_(self.lora_B_experts[expert_name].weight, a=math.sqrt(5))
            nn.init.kaiming_uniform_(self.lora_A_experts[expert_name].weight, a=math.sqrt(5))
        self.lora_B_shared = nn.Linear(self.shared_rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A_shared.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B_shared.weight)
        self.lora_dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, hidden_dim = x.shape
        original_output = self.original_layer(x)
        x_flat = x.reshape(-1, hidden_dim)
        router_logits = self.router(x_flat)
        routing_weights_before = F.softmax(router_logits, dim=1, dtype=torch.float)
        routing_weights, selected_experts = torch.topk(routing_weights_before, self.top_k, dim=-1)
        routing_weights /= routing_weights.sum(dim=-1, keepdim=True)
        routing_weights = routing_weights.to(x_flat.dtype)
        expert_mask = F.one_hot(selected_experts, num_classes=self.num_experts).permute(2, 1, 0)
        x_lora = self.lora_A_shared(self.lora_dropout(x_flat))
        combined_expert_output = torch.zeros((batch_size * sequence_length, self.shared_rank), dtype=x_lora.dtype, device=x_lora.device)
        for expert_idx in range(self.num_experts):
            idx, top_x = torch.where(expert_mask[expert_idx])
            if top_x.shape[0] == 0: continue
            expert_input = x_lora[top_x]
            expert_output = self.lora_A_experts[str(expert_idx)](self.lora_B_experts[str(expert_idx)](expert_input))
            current_expert_output = expert_output * routing_weights[top_x, idx, None]
            combined_expert_output.index_add_(0, top_x, current_expert_output.to(x_lora.dtype))
        final_lora_output = self.lora_B_shared(combined_expert_output)
        final_lora_output = final_lora_output.reshape(batch_size, sequence_length, -1)
        final_lora_output = final_lora_output * self.scaling
        if self.training:
            P = routing_weights_before
            imp = P.mean(dim=0)
            with torch.no_grad():
                assign_counts = torch.bincount(selected_experts.reshape(-1), minlength=self.num_experts).float()
                load = assign_counts / assign_counts.sum().clamp_min(1.0)
            self._lb_loss = self.num_experts * (imp * load).sum()
        else:
            self._lb_loss = None
        return original_output + final_lora_output

class LBTrainer(Trainer):
    def __init__(self, *args, lb_coef: float = 1e-2, **kwargs):
        super().__init__(*args, **kwargs)
        self.lb_coef = lb_coef
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        loss, outputs = super().compute_loss(model, inputs, return_outputs=True, **kwargs)
        aux_loss = None
        for m in model.modules():
            if hasattr(m, "_lb_loss"):
                lb = getattr(m, "_lb_loss", None)
                if lb is not None:
                    aux_loss = lb if aux_loss is None else (aux_loss + lb)
        if aux_loss is not None and self.is_in_train:
            loss = loss + self.lb_coef * aux_loss
        return (loss, outputs) if return_outputs else loss


def patch_roberta_with_edge_aware_lora(model, **kwargs):
    for layer in model.encoder.layer:
        layer.attention.self.query = EdgeAwareLoRALinear(layer.attention.self.query, **kwargs)
        layer.attention.self.value = EdgeAwareLoRALinear(layer.attention.self.value, **kwargs) # <-- FIX WAS HERE
    return model

# --- Setup ---
set_seed(42)
MODEL_NAME = "roberta-base"
LABELS = ["NEGATIVE", "POSITIVE"]
tricky_sentence = "This movie was not bad at all."

# --- Helper function to make predictions ---
def predict(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred_idx = torch.argmax(logits, dim=-1).item()
    return LABELS[pred_idx]

# --- Step 1: Show the "General" Model Failing ---
print("--- 1. Testing General Model (roberta-base) ---")
general_tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
general_model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to("cuda")
general_model.eval()

total_params = sum(p.numel() for p in general_model.parameters())
print(f"Total parameters in base model: {total_params:,}")

pred_before = predict(general_model, general_tokenizer, tricky_sentence)
print(f"Sentence: '{tricky_sentence}'")
print(f"Prediction BEFORE personalization: {pred_before}")

# --- Step 2: Define "Personalization" Data ---
print("\n--- 2. Creating tiny 'personal' dataset ---")
personal_data = {
    'sentence': [
        "This was not bad.", "That was not terrible.", "I'm not unhappy.",
        "It's not the worst.", "That's pretty good.", "I liked it a lot.",
        "This was not bad at all.", "Not a terrible movie.", "He isn't wrong.",
        "This is a great film.", "The service was not bad.", "The food was not terrible."
    ],
    'labels': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1] # All are POSITIVE
}
personal_dataset = Dataset.from_dict(personal_data)

def tokenize(example):
    return general_tokenizer(example["sentence"], truncation=True, padding="max_length", max_length=64)
personal_dataset = personal_dataset.map(tokenize, batched=True, remove_columns=["sentence"])
personal_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# --- Step 3: Patch the Model with Edge-Aware LoRA ---
print("\n--- 3. Patching model with Edge-Aware LoRA ---")
personal_model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
personal_model.roberta = patch_roberta_with_edge_aware_lora(
    personal_model.roberta, num_experts=2, top_k=1, shared_rank=4, expert_hidden_dim=16
)
for name, param in personal_model.named_parameters():
    if not any(x in name for x in ["router", "lora_A_shared", "lora_B_shared", "lora_B_experts", "lora_A_experts"]):
        param.requires_grad = False
    else:
        param.requires_grad = True

print("Model patched. Calculating parameter breakdown...")

# --- Step 4: Rapid "On-Device" Fine-Tuning ---
print("\n--- 4. Starting rapid personalization (fine-tuning)... ---")
training_args = TrainingArguments(
    output_dir="./personalization-demo",
    per_device_train_batch_size=2,
    num_train_epochs=50,
    learning_rate=1e-4,
    logging_steps=10,
    report_to="none",
    fp16=True,
    seed=42
)

trainer = LBTrainer(
    model=personal_model,
    args=training_args,
    train_dataset=personal_dataset
)

trainer.train()
print("--- Personalization (fine-tuning) COMPLETE ---")

# --- Step 5: Show the "Personalized" Model Succeeding ---
print("\n--- 5. Testing Personalized Model ---")
personal_model.to("cuda")
personal_model.eval()

pred_after = predict(personal_model, general_tokenizer, tricky_sentence)
print(f"Sentence: '{tricky_sentence}'")
print(f"Prediction AFTER personalization: {pred_after}")


print("\n--- 6. Efficiency Analysis ---")
# --- Split trainable params into Edge vs Cloud ---
edge_only_params = 0
cloud_trainable_params = 0
edge_state_dict = {}

for name, param in personal_model.named_parameters():
    if param.requires_grad:
        # According to your design, router and experts are on the edge
        if "router" in name or "lora_A_experts" in name or "lora_B_experts" in name:
            edge_only_params += param.numel()
            edge_state_dict[name] = param.data
        else:
            # A_shared and B_shared are the "Cloud" components
            cloud_trainable_params += param.numel()

total_trainable_params = edge_only_params + cloud_trainable_params

# --- Calculate percentages ---
edge_percent_of_total = (edge_only_params / total_params) * 100
cloud_percent_of_total = (cloud_trainable_params / total_params) * 100
total_trainable_percent = (total_trainable_params / total_params) * 100

# --- Get file sizes ---
full_model_path = "full_base_model.pth"
torch.save(general_model.state_dict(), full_model_path)
full_model_size_mb = os.path.getsize(full_model_path) / (1024 * 1024)

edge_weights_path = "edge_weights_ONLY.pth"
torch.save(edge_state_dict, edge_weights_path)
edge_weights_size_mb = os.path.getsize(edge_weights_path) / (1024 * 1024)

print("\n" + "="*52)
print("     🚀 PERSONALIZE-ON-EDGE DEMO (roberta-base) 🚀")
print("="*52)

print("\n--- PERFORMANCE ---")
print(f"Sentence:            '{tricky_sentence}'")
print(f"General Model:        {pred_before}")
print(f"Personalized Model:   {pred_after}")

print("\n--- EFFICIENCY ---")
print(f"Total Base Parameters:   {total_params:,}")

print("\n--- PARAMETER BREAKDOWN (Our LoRA-MoE) ---")
print(f"TOTAL Trainable:     {total_trainable_params:,} ({total_trainable_percent:.3f}% of total)")
print("--------------------------------------------------")
print(f"  > On-Device 'Edge':  {edge_only_params:,} ({edge_percent_of_total:.4f}%)")
print(f"  > 'Cloud' (Shared):  {cloud_trainable_params:,} ({cloud_percent_of_total:.4f}%)")

print("\n--- STORAGE FOOTPRINT ---")
print(f"Full Model Storage:      {full_model_size_mb:.2f} MB")
print(f"ON-DEVICE Storage:       {edge_weights_size_mb:.2f} MB   <-- (This is what the user stores!)")
print("="*52)

if pred_before != pred_after:
    print("\n The model adapted to the user's personal style.")

# Clean up the large files
os.remove(full_model_path)
os.remove(edge_weights_path)